# LeetCode #1330: Reverse Subarray To Maximize Array Value

https://leetcode.com/problems/reverse-subarray-to-maximize-array-value/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(1)$ |
| **Optimal: Running-Extremes Scan ★** | $O(n)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Try every subarray $[l, r]$, compute the change in array value (only the two boundary pairs change when a subarray is reversed), and track the maximum. The $O(n^2)$ pairs make this too slow for $n = 3 \times 10^4$.

### Optimal: Running-Extremes Scan ★
The array value = $\sum |\text{arr}[i] - \text{arr}[i+1]|$. Reversing $[l, r]$ only changes the two boundary pairs; the inner terms cancel. The gain equals $|a - d| + |b - c| - |a - b| - |c - d|$ where $(a,b)$ and $(c,d)$ are the boundary pairs. Expanding the absolute values into 6 cases yields 6 running extrema (max/min of individual elements and differences of consecutive pairs) that can be tracked in one forward scan, giving $O(n)$ overall.

**Constraints:**
* $1 \leq n \leq 3 \times 10^4$
* $-10^9 \leq \text{arr}[i] \leq 10^9$


## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxValueAfterReverse(int[] arr) {
        int n = arr.Length;
        // Compute the base array value
        int baseVal = 0;
        for (int i = 0; i < n - 1; i++)
            baseVal += Math.Abs(arr[i] - arr[i + 1]);

        int maxGain = 0;

        // Edge case: reverse [0, r] — only right boundary pair changes
        for (int r = 1; r < n - 1; r++)
            maxGain = Math.Max(maxGain,
                Math.Abs(arr[0] - arr[r + 1]) - Math.Abs(arr[r] - arr[r + 1]));

        // Edge case: reverse [l, n-1] — only left boundary pair changes
        for (int l = 1; l < n - 1; l++)
            maxGain = Math.Max(maxGain,
                Math.Abs(arr[l - 1] - arr[n - 1]) - Math.Abs(arr[l - 1] - arr[l]));

        // General case: both boundaries exist — track 6 running extrema
        // For boundary pairs (a,b)=(arr[i],arr[i+1]) left and (c,d)=(arr[j],arr[j+1]) right
        // gain = |a-d|+|b-c|-|a-b|-|c-d|, expanded via sign cases into:
        // max(2*maxB-c-d, 2*maxA-c-d, c+d-2*minA, c+d-2*minB, 2*maxAmB+c-d, 2*maxBmA+d-c) - |c-d|
        if (n >= 4) {
            long maxA = arr[0], maxB = arr[1];
            long minA = arr[0], minB = arr[1];
            long maxAmB = arr[0] - arr[1], maxBmA = arr[1] - arr[0];

            for (int j = 1; j < n - 2; j++) {
                long c = arr[j + 1], d = arr[j + 2];
                long absCD = Math.Abs(c - d);
                long gain = Math.Max(0, Math.Max(
                    Math.Max(2 * maxB - c - d - absCD, 2 * maxA - c - d - absCD),
                    Math.Max(
                        Math.Max(c + d - 2 * minA - absCD, c + d - 2 * minB - absCD),
                        Math.Max(2 * maxAmB + c - d - absCD, 2 * maxBmA + d - c - absCD))));
                if (gain > maxGain) maxGain = (int)gain;

                // Add new left pair (arr[j], arr[j+1]) for future right pairs
                long na = arr[j], nb = arr[j + 1];
                if (na > maxA) maxA = na; if (na < minA) minA = na;
                if (nb > maxB) maxB = nb; if (nb < minB) minB = nb;
                if (na - nb > maxAmB) maxAmB = na - nb;
                if (nb - na > maxBmA) maxBmA = nb - na;
            }
        }

        return baseVal + maxGain;
    }
}

### Python

In [ ]:
class Solution:
    def maxValueAfterReverse(self, arr: list[int]) -> int:
        n = len(arr)
        # Compute the base array value
        base_val = sum(abs(arr[i] - arr[i+1]) for i in range(n - 1))
        max_gain = 0

        # Edge case: reverse [0, r] — only right boundary pair changes
        for r in range(1, n - 1):
            max_gain = max(max_gain,
                abs(arr[0] - arr[r+1]) - abs(arr[r] - arr[r+1]))

        # Edge case: reverse [l, n-1] — only left boundary pair changes
        for l in range(1, n - 1):
            max_gain = max(max_gain,
                abs(arr[l-1] - arr[n-1]) - abs(arr[l-1] - arr[l]))

        # General case: track 6 running extrema across left boundary pairs
        if n >= 4:
            max_a = max_b = arr[0]
            min_a = min_b = arr[0]
            max_a, min_a = arr[0], arr[0]
            max_b, min_b = arr[1], arr[1]
            max_amb = arr[0] - arr[1]
            max_bma = arr[1] - arr[0]

            for j in range(1, n - 2):
                c, d = arr[j+1], arr[j+2]
                abs_cd = abs(c - d)
                gain = max(0,
                    2*max_b - c - d - abs_cd,
                    2*max_a - c - d - abs_cd,
                    c + d - 2*min_a - abs_cd,
                    c + d - 2*min_b - abs_cd,
                    2*max_amb + c - d - abs_cd,
                    2*max_bma + d - c - abs_cd)
                max_gain = max(max_gain, gain)

                # Add new left pair (arr[j], arr[j+1]) for future right pairs
                na, nb = arr[j], arr[j+1]
                max_a = max(max_a, na); min_a = min(min_a, na)
                max_b = max(max_b, nb); min_b = min(min_b, nb)
                max_amb = max(max_amb, na - nb)
                max_bma = max(max_bma, nb - na)

        return base_val + max_gain


### Go

In [ ]:
func maxValueAfterReverse(arr []int) int {
    n := len(arr)
    // Compute the base array value
    baseVal := 0
    for i := 0; i < n-1; i++ {
        baseVal += abs1330(arr[i] - arr[i+1])
    }
    maxGain := 0

    // Edge case: reverse [0, r]
    for r := 1; r < n-1; r++ {
        g := abs1330(arr[0]-arr[r+1]) - abs1330(arr[r]-arr[r+1])
        if g > maxGain { maxGain = g }
    }
    // Edge case: reverse [l, n-1]
    for l := 1; l < n-1; l++ {
        g := abs1330(arr[l-1]-arr[n-1]) - abs1330(arr[l-1]-arr[l])
        if g > maxGain { maxGain = g }
    }

    // General case: track 6 running extrema
    if n >= 4 {
        maxA, minA := arr[0], arr[0]
        maxB, minB := arr[1], arr[1]
        maxAmB := arr[0] - arr[1]
        maxBmA := arr[1] - arr[0]

        for j := 1; j < n-2; j++ {
            c, d := arr[j+1], arr[j+2]
            absCD := abs1330(c - d)
            candidates := []int{
                2*maxB - c - d - absCD,
                2*maxA - c - d - absCD,
                c + d - 2*minA - absCD,
                c + d - 2*minB - absCD,
                2*maxAmB + c - d - absCD,
                2*maxBmA + d - c - absCD,
            }
            for _, g := range candidates {
                if g > maxGain { maxGain = g }
            }
            // Add new left pair
            na, nb := arr[j], arr[j+1]
            if na > maxA { maxA = na }; if na < minA { minA = na }
            if nb > maxB { maxB = nb }; if nb < minB { minB = nb }
            if na-nb > maxAmB { maxAmB = na - nb }
            if nb-na > maxBmA { maxBmA = nb - na }
        }
    }

    return baseVal + maxGain
}

func abs1330(x int) int {
    if x < 0 { return -x }
    return x
}

### Rust

In [ ]:
impl Solution {
    pub fn max_value_after_reverse(arr: Vec<i32>) -> i32 {
        let n = arr.len();
        // Compute the base array value
        let base_val: i64 = (0..n-1).map(|i| (arr[i] - arr[i+1]).abs() as i64).sum();
        let mut max_gain: i64 = 0;

        // Edge case: reverse [0, r]
        for r in 1..n-1 {
            let g = (arr[0] - arr[r+1]).abs() as i64 - (arr[r] - arr[r+1]).abs() as i64;
            if g > max_gain { max_gain = g; }
        }
        // Edge case: reverse [l, n-1]
        for l in 1..n-1 {
            let g = (arr[l-1] - arr[n-1]).abs() as i64 - (arr[l-1] - arr[l]).abs() as i64;
            if g > max_gain { max_gain = g; }
        }

        // General case: track 6 running extrema
        if n >= 4 {
            let (mut max_a, mut min_a) = (arr[0] as i64, arr[0] as i64);
            let (mut max_b, mut min_b) = (arr[1] as i64, arr[1] as i64);
            let mut max_amb = (arr[0] - arr[1]) as i64;
            let mut max_bma = (arr[1] - arr[0]) as i64;

            for j in 1..n-2 {
                let (c, d) = (arr[j+1] as i64, arr[j+2] as i64);
                let abs_cd = (c - d).abs();
                let gain = [0,
                    2*max_b - c - d - abs_cd,
                    2*max_a - c - d - abs_cd,
                    c + d - 2*min_a - abs_cd,
                    c + d - 2*min_b - abs_cd,
                    2*max_amb + c - d - abs_cd,
                    2*max_bma + d - c - abs_cd,
                ].into_iter().max().unwrap();
                if gain > max_gain { max_gain = gain; }

                let (na, nb) = (arr[j] as i64, arr[j+1] as i64);
                if na > max_a { max_a = na; } if na < min_a { min_a = na; }
                if nb > max_b { max_b = nb; } if nb < min_b { min_b = nb; }
                if na - nb > max_amb { max_amb = na - nb; }
                if nb - na > max_bma { max_bma = nb - na; }
            }
        }

        (base_val + max_gain) as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [2, 3, 1, 5, 4]`
Base value = $|2-3|+|3-1|+|1-5|+|5-4| = 1+2+4+1 = 8$. Reversing `[1, 4]` gives `[2, 4, 5, 1, 3]`... the running-extremes scan finds the best gain of 2, returning 10.

### 2. Slightly Complex
**Input:** `arr = [3, 1, 2, 4]`
Base = 5. The edge case `reverse [2, 3]` gains 2 (right-boundary edge scan): $|arr[1] - arr[3]| - |arr[1] - arr[2]| = 3 - 1 = 2$. Result: 7.

### 3. Edge Case: Time Factor
**Input:** `arr = [1, 2, 3, \ldots, 30000]` (strictly increasing, $n = 3 \times 10^4$)
Base value = $n - 1 = 29{,}999$. No reversal improves consecutive differences when the array is already sorted — the scan runs all $O(n)$ steps and correctly returns 0 gain.

### 4. Edge Case: Space Factor
**Input:** `arr = [5, 5, 5, 5]`
All differences are 0; base = 0 and no reversal changes anything — gain = 0. The algorithm uses only 6 scalar variables beyond the input array, confirming $O(1)$ extra space.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [-10^9, 10^9, -10^9, 10^9]`
Base = $3 \times 2 \times 10^9 = 6 \times 10^9$ — overflows 32-bit integers. The solution uses `long`/`i64` intermediates to handle values up to $\approx 4 \times 10^{18}$, then casts the final result which fits in int32 after the modular gain is applied.
